***MGSC 662 - Assignment 1***

***Fresh Milk Transportation***

*BalancedMilk* is an old dairy-transportation firm renowned for the freshness of the milk it distributes. It has long-term fixed-amount contracts with eight dairy farms (suppliers) and distributes their production among ten demand markets. Transporting milk for a long time from suppliers to demand markets, BalancedMilk is well aware of the milk demand in each demand market and the transportation cost from each supplier to each demand center.

In the following table, the number in front of each supply and demand center represents the supply and demand (in tonnes.) Also, for each supply-demand pair, the transportation cost ($ per tonne) is depicted.

| | $D_1$(90)| $D_2$(100) | $D_3$(150) | $D_4$(190) | $D_5$(180) | $D_6$(240) | $D_7$(210) | $D_8$(90) | $D_9$(160) | $D_0$(70) |
|--| -- | -- | -- | -- | -- | -- | -- | -- | -- | -- |
| $S_1$(110) |20|49|16|30|8|35|21|40|10|12|
| $S_2$(80) |15|53|7|20|47|11|16|17|15|44|
| $S_3$(100) |22|25|42|22|31|9|11|29|20|5|
| $S_4$(240) |45|6|33|35|49|25|30|47|32|31|
| $S_5$(100) |9|12|41|15|38|14|53|22|12|13|
| $S_6$(280) |21|24|32|49|5|47|30|23|37|8|
| $S_7$(130) |12|19|5|28|47|39|15|35|9|51|
| $S_8$(440) |34|17|10|21|9|33|14|26|19|45|

Keep the following points in mind when solving the problems:

**Note 1**: The firm should not supply milk to a demand market **more** than its demand, i.e., should not **over-supply** it.

**Note 2**: Write your code so that your answer to each part is independent of the other. For instance, to get the solution for Problem 2, one should not be forced to run Problem 1 beforehand. 

For each of the following strategies, formulate them as a LP problem and solve them using Gurobi:

**Problem 1:**

The firm wants to distribute all the supplied milk as efficiently as possible. Formulate the problem of minimizing the total transportation cost using Gurobi and solve it. Then, report the optimal allocation and the minimum cost.

In [15]:
Cost1 = [[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
        [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
        [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
        [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
        [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
        [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
        [12, 19,  5, 28, 47, 39, 15, 35, 9, 51],
        [34, 17, 10, 21, 9, 33, 14, 26, 19, 45]]

SupplyCenter1 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8']
Supply1 = [110, 80, 100, 240, 100, 280, 130, 440]
DemandMarket1 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand1 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd

#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()


#setup problem
prob_1=gb.Model("Question 1", env=env)

#create variables to define matrix of variables
Slen1=len(Supply1)
Dlen1=len(Demand1)

#Create variables for each demand and supply
X1=prob_1.addVars(Slen1,Dlen1,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter1 for j in DemandMarket1])

#set objective function 
prob_1.setObjective(sum(Cost1[i][j]*X1[i,j] for i in range(Slen1)for j in range (Dlen1)),GRB.MINIMIZE)

#set constraints
##setting the supply constrains for each supply centre
for i in range(Slen1):
    prob_1.addConstr(sum(X1[i,j] for j in range (Dlen1))<=Supply1[i])
##setting the demand constrains for each demand centre with no over supply of demand
for j in range(Dlen1):
    prob_1.addConstr(sum(X1[i,j] for i in range (Slen1))==Demand1[j])
    
#optimize problem
prob_1.optimize()
   
#create table and data frame for results
pb1dict={}
l=[]
for i in X1:
    col=("D_"+str(i[1]+1))
    if col in pb1dict.keys():
        pb1dict[col].append(X1[i].X)
    else:
        l1=[]
        pb1dict[col]=l1
        pb1dict[col].append(X1[i].X)

df1=pd.DataFrame(pb1dict,index=SupplyCenter1)
print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df1)

#print objective cost
print("\n")
print("The optimized minmum total cost is therefore ${:,}".format(round(prob_1.objVal)))



The optimized supply and demand is visualized in a table below:
      D_1    D_2   D_3    D_4    D_5    D_6    D_7   D_8    D_9  D_10
S_1   0.0    0.0   0.0    0.0    0.0    0.0    0.0   0.0  110.0   0.0
S_2   0.0    0.0  20.0    0.0    0.0    0.0    0.0  60.0    0.0   0.0
S_3   0.0    0.0   0.0    0.0    0.0  100.0    0.0   0.0    0.0   0.0
S_4   0.0  100.0   0.0    0.0    0.0  140.0    0.0   0.0    0.0   0.0
S_5  90.0    0.0   0.0   10.0    0.0    0.0    0.0   0.0    0.0   0.0
S_6   0.0    0.0   0.0    0.0  180.0    0.0    0.0  30.0    0.0  70.0
S_7   0.0    0.0  80.0    0.0    0.0    0.0    0.0   0.0   50.0   0.0
S_8   0.0    0.0  50.0  180.0    0.0    0.0  210.0   0.0    0.0   0.0


The optimized minmum total cost is therefore $18,440


Due to a cattle virus outbreak in the largest supplier ($S_8$), the health ministry shut down this dairy farm. Left with seven suppliers and unchanged demands, BalancedMilk is facing a supply shortage. Therefore, the management establishes two teams to search for short-term and long-term strategies to combat this issue. While the *long-term team* is looking for alternative suppliers or new contracts, we want to evaluate the strategies offered by the *short-term team* in the following problems.

**Problem 2:**

Suppose not supplying milk to a demand market is costless. Nevertheless, the firm should supply all the milk the remaining seven supply centers provide. Also, remember that the firm cannot oversupply a demand market. Formulate the problem of minimizing the total transportation cost using Gurobi and solve it. Then, report the optimal allocation and the minimum cost. 

(Hint: one way of modeling this situation is to replace $S_8$ with a *dummy supplier* that provides the same amount of milk previously provided by $S_8$ but at zero cost.)


In [16]:
Cost2 = [[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
        [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
        [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
        [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
        [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
        [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
        [12, 19,  5, 28, 47, 39, 15, 35, 9, 51],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

SupplyCenter2 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'Dummy Supply']
Supply2 = [110, 80, 100, 240, 100, 280, 130, 440]
DemandMarket2 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand2 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd

#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()

#setup problem
prob_2=gb.Model("Question 2",env=env)


#Create variables to define matrix of variables
Slen2=len(Supply2)
Dlen2=len(Demand2)

#Create variables for each demand and supply
X2=prob_2.addVars(Slen2,Dlen2,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter2 for j in DemandMarket2])

#set objective function 
prob_2.setObjective(sum(Cost2[i][j]*X2[i,j] for i in range(Slen2)for j in range (Dlen2)),GRB.MINIMIZE)

#set constraints
##setting the supply constrains for each supply centre
for i in range(Slen2):
    prob_2.addConstr(sum(X2[i,j] for j in range (Dlen2))<=Supply2[i])
##setting the demand constrains for each demand centre with no over supply of demand
for j in range(Dlen2):
    prob_2.addConstr(sum(X2[i,j] for i in range (Slen2))==Demand2[j])
    
#optimize problem
prob_2.optimize()

#Create dictionary for datafram and result printing ease
pb2dict={}

#create dictionary for matrix output
for i in X2:
    col=("D_"+str(i[1]+1))

    if col in pb2dict.keys():
        pb2dict[col].append(X2[i].X)
    else:
        l2=[]
        pb2dict[col]=l2
        pb2dict[col].append(X2[i].X)

#insert data dictionary to dataframe      
df2=pd.DataFrame(pb2dict,index=SupplyCenter2)
print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df2)

#print objective cost
print("\n")
print("The optimized minmum total cost is therefore ${:,}".format(round(prob_2.objVal)))



The optimized supply and demand is visualized in a table below:
               D_1    D_2   D_3    D_4    D_5    D_6    D_7   D_8    D_9  D_10
S_1            0.0    0.0   0.0    0.0    0.0    0.0    0.0   0.0  110.0   0.0
S_2            0.0    0.0  60.0    0.0    0.0   20.0    0.0   0.0    0.0   0.0
S_3            0.0    0.0   0.0    0.0    0.0   80.0   20.0   0.0    0.0   0.0
S_4            0.0  100.0   0.0    0.0    0.0  140.0    0.0   0.0    0.0   0.0
S_5           90.0    0.0   0.0    0.0    0.0    0.0    0.0   0.0   10.0   0.0
S_6            0.0    0.0   0.0    0.0  180.0    0.0    0.0  30.0    0.0  70.0
S_7            0.0    0.0  90.0    0.0    0.0    0.0    0.0   0.0   40.0   0.0
Dummy Supply   0.0    0.0   0.0  190.0    0.0    0.0  190.0  60.0    0.0   0.0


The optimized minmum total cost is therefore $10,670


**Problem 3:**

In the real world, with fierce competitors ready to attack *BalancedMilk*'s market share, the cost of not supplying milk to a demand center is not zero. Additionally, like Problem 2, the firm should supply all the milk provided by the remaining seven supply centers. Suppose the cost of not delivering each tonne of milk to each demand center is  20\\$/tone. Formulate the problem of minimizing the sum of total transportation and not supplying costs using Gurobi and solve it. Then, report the optimal allocation and the minimum cost. Now suppose instead of 20\\$/tone, the not supplying cost is 100\$/tone. Again, formulate the problem of minimizing the sum of total transportation and not supplying costs using Gurobi and solve it. Then, report the optimal allocation and the minimum cost. 

Can you detect any patterns in the optimal allocation in this problem and the previous one? How about the difference between the minimum costs? Explain your intuition.

In [17]:
Cost3 = [[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
        [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
        [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
        [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
        [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
        [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
        [12, 19,  5, 28, 47, 39, 15, 35, 9, 51]]

SupplyCenter3 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'Dummy Supply']
Supply3 = [110, 80, 100, 240, 100, 280, 130, 440]
DemandMarket3 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand3 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd
import copy

#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()

#set function for repeated attempts
def Re_optimize3(SupplyCenter,Supply,DemandMarket,Demand,probn,Cost):
        
    #create variables to define matrix of variables
    Slen=len(Supply)
    Dlen=len(Demand)
    #Create variables for each demand and supply
    X=probn.addVars(Slen,Dlen,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter for j in DemandMarket])

    #set objective function 
    probn.setObjective(sum(Cost[i][j]*X[i,j] for i in range(Slen)for j in range (Dlen)),GRB.MINIMIZE)

    #set constraints
    ##setting the supply constrains for each supply centre
    for i in range(Slen):
        probn.addConstr(sum(X[i,j] for j in range (Dlen))<=Supply[i])
    ##setting the demand constrains for each demand centre with no over supply of demand
    for j in range(Dlen):
        probn.addConstr(sum(X[i,j] for i in range (Slen))==Demand[j])

    #optimize problem
    probn.optimize()

    #create dictionary for matrix output
    pbndict={}
    l=[]
    for i in X:
        col=("D_"+str(i[1]+1))
        if col in pbndict.keys():
            pbndict[col].append(X[i].X)
        else:
            l=[]
            pbndict[col]=l
            pbndict[col].append(X[i].X)
    #Title="Optimized Supply"
    df=pd.DataFrame(pbndict,index=SupplyCenter)
    return df,probn


#setup problem
prob_3a=gb.Model("Question 3A" , env=env)
prob_3b=gb.Model("Question 3B", env=env)

#append supply penalty cost
Cost3a=copy.deepcopy(Cost3)
Cost3a.append([20, 20, 20, 20, 20, 20, 20, 20, 20, 20])
Cost3b=copy.deepcopy(Cost3)
Cost3b.append([100, 100, 100, 100, 100, 100, 100, 100, 100, 100])

#Call function Re_optimize(SupplyCenter,Supply,DemandMarket,Demand,prob_3,Cost3)
df3a,prob_3a=Re_optimize3(SupplyCenter3,Supply3,DemandMarket3,Demand3,prob_3a,Cost3a)
df3b,prob_3b=Re_optimize3(SupplyCenter3,Supply3,DemandMarket3,Demand3,prob_3b,Cost3b)

print("The optimized supply and demand is visualized in a table below:")
#print("\n")
print(df3a)

#print objective cost for 3a
print("\n")
print("The optimized minmum total cost, with a cost of $20/tonne for not delivering is therefore ${:,}".format(round(prob_3a.objVal)))
print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df3b)
print("\n")
#print objective cost for 3b
print("The optimized minmum total cost, with a cost of $100/tonne for not delivering is therefore ${:,}".format(round(prob_3b.objVal)))



The optimized supply and demand is visualized in a table below:
               D_1    D_2   D_3    D_4    D_5    D_6    D_7   D_8    D_9  D_10
S_1            0.0    0.0   0.0    0.0    0.0    0.0    0.0   0.0  110.0   0.0
S_2            0.0    0.0  60.0    0.0    0.0   20.0    0.0   0.0    0.0   0.0
S_3            0.0    0.0   0.0    0.0    0.0   80.0   20.0   0.0    0.0   0.0
S_4            0.0  100.0   0.0    0.0    0.0  140.0    0.0   0.0    0.0   0.0
S_5           90.0    0.0   0.0    0.0    0.0    0.0    0.0   0.0   10.0   0.0
S_6            0.0    0.0   0.0    0.0  180.0    0.0    0.0  30.0    0.0  70.0
S_7            0.0    0.0  90.0    0.0    0.0    0.0    0.0   0.0   40.0   0.0
Dummy Supply   0.0    0.0   0.0  190.0    0.0    0.0  190.0  60.0    0.0   0.0


The optimized minmum total cost, with a cost of $20/tonne for not delivering is therefore $19,470


The optimized supply and demand is visualized in a table below:
               D_1    D_2   D_3    D_4    D_5    D_6    D_7

### <font color=#FF0000> **Insight:** </font>
*Can you detect any patterns in the optimal allocation in this problem and the previous one? How about the difference between the minimum costs? Explain your intuition.*

The optimal allocation between problems #2 and #3 **remains the same** for the suppliers and the demand. 

The optimal allocation remaining constant makes sense as the only variation between these problems is the cost of not delivering the supply from the dummy variable S8. Meanwhile all the other costs for delivery and other model constraints remain the same between problems #2 and #3. Therefore it would make sense that the two model minimums are the same and would allocate the same distribtion. The minimum for the delivered supply is therefore constant with only the cost of not delivering changing.

The only difference in the model is the penalty costs, just like the difference between the minimum costs is a function of the penalty cost. Specifically *440 X pc where 440 is the non-delivered supply and pc is the penalty cost*. 

The function of cost makes sense as the penatly cost is the only variable that is changing through these models. Therefore the predicted totla minum cost is equal to mincost= *\\$10,670 + 440 X pc.* We can validate the answers to problems #2 and #3 with this function.


*Min Cost Prob 2  = 10,670 + 440 X 0   = 10,670 + 0      = \\$10,670* **(matches model output)**

*Min Cost Prob 3a = 10,670 + 440 X 20  = 10,670 + 8,800  = \\$19,470* **(matches model output)**

*Min Cost prob 3b = 10,670 + 440 X 100 = 10,670 + 44,000 = \\$54,670*  **(matches model output)**

**Problem 4:**

In Problems 2 and 3, we assumed the firm is committed to distributing all the milk supplied by the seven remaining supply centers. Suppose the firm does not entertain this constraint anymore, and the cost of not supplying each tonne of milk to each demand center is  20\\$/tone. Formulate the problem of minimizing the sum of total transportation and not supplying costs using Gurobi and solve it. Then, report the optimal allocation and the minimum cost. Now suppose instead of 20\\$/tone, the not supplying cost is 100\$/tone. Again, formulate the problem of minimizing the sum of total transportation and not supplying costs using Gurobi and solve it. Then, report the optimal allocation and the minimum cost. 

Do you see any difference between the optimal allocation in this problem and Problem 3? Do you have any educated guesses on why it is the case?

In [18]:
Cost4 = [[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
        [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
        [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
        [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
        [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
        [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
        [12, 19,  5, 28, 47, 39, 15, 35, 9, 51]]


SupplyCenter4 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'Summy Supply']
Supply4 = [110, 80, 100, 240, 100, 280, 130, 440]
DemandMarket4 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand4 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

# Your solution for Problem 1:
#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd
import copy

#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()

#set function re_optimize for ease of repeated attempts
def Re_optimize4(SupplyCenter,Supply,DemandMarket,Demand,probn,Cost):
        #create variables to define matrix of variables
    Slen=len(Supply)
    Dlen=len(Demand)
    #print(SupplyCenter,Supply,DemandMarket,Demand,probn,Cost)
    #Create variables for each demand and supply
    X=probn.addVars(Slen,Dlen,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter for j in DemandMarket])
    
    #set objective function 
    probn.setObjective(sum(Cost[i][j]*X[i,j] for i in range(Slen)for j in range (Dlen)),GRB.MINIMIZE)

    #set constraints
    ##setting the supply constrains for each supply centre
    for i in range(Slen):
        probn.addConstr(sum(X[i,j] for j in range (Dlen))<=Supply[i])
    ##setting the demand constrains for each demand centre with no over supply of demand
    for j in range(Dlen):
        probn.addConstr(sum(X[i,j] for i in range (Slen))==Demand[j])

    #optimize problem
    probn.optimize()
    
    #create dictionary that will be converted to matrix for results display
    pbndict={}
    l=[]
    for i in X:
        col=("D_"+str(i[1]+1))
        if col in pbndict.keys():
            pbndict[col].append(X[i].X)
        else:
            l=[]
            pbndict[col]=l
            pbndict[col].append(X[i].X)
            
    df=pd.DataFrame(pbndict,index=SupplyCenter)
    return df,probn

#if company doe not have to deliveriver to demand centres
#at a flat penalty cost for non_deliv 
#then no delvier where cost is higher than the penalty cost.
#Therefore replace the cost with penalty cost if penalty cost is lower
#append to supply 8 as regardless they will not supply

def min_cost_fn4(Cost,non_deliv):
    for i in range(len(Costn)):
        #if delivery cost is above non_deliv then change to not deliver at cost of non_deliv
        for j in range(len(Costn[i])):
            if int(Costn[i][j])>non_deliv:
                Costn[i][j]=non_deliv
            #else do nothing
    #append for supplier #8 who will not be delivering
    Costn.append([non_deliv, non_deliv, non_deliv, non_deliv, non_deliv, non_deliv, non_deliv, non_deliv, non_deliv, non_deliv])
    return Costn



#setup problem
prob_4a=gb.Model("Question 4A", env=env)
prob_4b=gb.Model("Question 4B", env=env)


#deepcopy required because of lists within lists copy challenge
Costn=copy.deepcopy(Cost4)
#modify lsit with function min_cost and append supply penalty cost
Cost4a=min_cost_fn4(Costn,20)
Costn=copy.deepcopy(Cost4)
Cost4b=min_cost_fn4(Costn,100)

#Re_optimize(SupplyCenter,Supply,DemandMarket,Demand,prob_3,Cost3)
#Re_optimize handles optimization of this problem with different variables i.e. cost
df4a,prob_4a=Re_optimize4(SupplyCenter4,Supply4,DemandMarket4,Demand4,prob_4a,Cost4a)
df4b,prob_4b=Re_optimize4(SupplyCenter4,Supply4,DemandMarket4,Demand4,prob_4b,Cost4b)

print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df4a)

#print objective cost
#print("\n")
print("The optimized minmum total cost, with a cost of $20/tonne for not delivering is therefore ${:,}".format(round(prob_4a.objVal)))


print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df4b)

#print objective cost
#print("\n")
print("The optimized minmum total cost, with a cost of $100/tonne for not delivering is therefore ${:,}".format(round(prob_4b.objVal)))



The optimized supply and demand is visualized in a table below:
               D_1    D_2   D_3    D_4    D_5    D_6    D_7   D_8    D_9  D_10
S_1            0.0    0.0   0.0    0.0    0.0    0.0    0.0   0.0  110.0   0.0
S_2            0.0    0.0  60.0    0.0    0.0   20.0    0.0   0.0    0.0   0.0
S_3            0.0    0.0   0.0    0.0    0.0  100.0    0.0   0.0    0.0   0.0
S_4            0.0  100.0   0.0    0.0    0.0   50.0    0.0  90.0    0.0   0.0
S_5           90.0    0.0   0.0    0.0    0.0    0.0    0.0   0.0   10.0   0.0
S_6            0.0    0.0   0.0    0.0  180.0   30.0    0.0   0.0    0.0  70.0
S_7            0.0    0.0  90.0    0.0    0.0    0.0    0.0   0.0   40.0   0.0
Summy Supply   0.0    0.0   0.0  190.0    0.0   40.0  210.0   0.0    0.0   0.0
The optimized minmum total cost, with a cost of $20/tonne for not delivering is therefore $18,640


The optimized supply and demand is visualized in a table below:
               D_1    D_2   D_3    D_4    D_5    D_6    D_7

### <font color=#FF0000> **Insight:** </font>
*Do you see any difference between the optimal allocation in this problem and Problem 3? Do you have any educated guesses on why it is the case?*

There are differences in the allocation between problem #4 and problem #3. Specifically for demand centers 6, 7 and 8. The added freedom of not having to supply all the produced milk to the demand centres alows the model to minimize costs by choosing between the penalty and cost to supply the demand. In other words, the cost of delivering from S6 to D6 is \\$47 hoever the model can choose to simply not deliver for a penalty of \\$20 and save \\$27 per unit in the process. Therefore it makes sesne to see a cost decrease.

**Problem 5:**

In Problems 2, 3 and 4, we assumed the no-supply cost is the same for all demand centers. However, this may not be the case. The estimations by the *short-term team* indicate that the cost of not supplying a tonne of milk to each demand center is as follows:

| | $D_1$| $D_2$ | $D_3$ | $D_4$ | $D_5$ | $D_6$ | $D_7$ | $D_8$ | $D_9$ | $D_{10}$|
|--| -- | -- | -- | -- | -- | -- | -- | -- | -- | -- |
| Cost of not supplying (\\$/tonne) |15|20|25|30|28|35|32|15|25|10|

Additionally, like Problem 2, the firm should supply all the milk provided by the remaining seven supply centers. Formulate the problem of minimizing the sum of total transportation and not supplying costs using Gurobi and then solve it. Report the optimal allocation and the minimum cost. Is the optimal allocation the same as Problem 2? Why?

In [19]:
#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd
import copy
import numpy as np

Cost5 = [[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
        [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
        [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
        [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
        [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
        [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
        [12, 19,  5, 28, 47, 39, 15, 35, 9, 51]]

Cost5.append([15, 20, 25, 30, 28, 35, 32, 15, 25, 10])

SupplyCenter5 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'Var Penalty']
Supply5 = [110, 80, 100, 240, 100, 280, 130, 440]
DemandMarket5 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand5 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

#create variables to define matrix of variables
Slen5=len(Supply5)
Dlen5=len(Demand5)

#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()

#setup problem
prob_5=gb.Model("Question 5", env=env)


#Create variables for each demand and supply
X5=prob_5.addVars(Slen5,Dlen5,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter5 for j in DemandMarket5])


#set objective function 
prob_5.setObjective(sum(Cost5[i][j]*X5[i,j] for i in range(Slen5)for j in range (Dlen5)),GRB.MINIMIZE)

#set constraints
##setting the supply constrains for each supply centre
for i in range(Slen5):
    prob_5.addConstr(sum(X5[i,j] for j in range (Dlen5))<=Supply5[i])
##setting the demand constrains for each demand centre with no over supply of demand
for j in range(Dlen5):
    prob_5.addConstr(sum(X5[i,j] for i in range (Slen5))==Demand5[j])
    
    
#print(X)
#optimize problem
prob_5.optimize()

pb5dict={}
l5=[]
for i in X5:
    col=("D_"+str(i[1]+1))      
    if col in pb5dict.keys():
        pb5dict[col].append(X5[i].X)
    else:
        l5=[]
        pb5dict[col]=l5
        pb5dict[col].append(X5[i].X)


df=pd.DataFrame(pb5dict,index=SupplyCenter5)
print("\n")
print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df)

#print objective cost
print("\n")
print("The optimized minmum total cost is therefore ${:,}".format(round(prob_5.objVal)))





The optimized supply and demand is visualized in a table below:
              D_1    D_2    D_3    D_4    D_5    D_6    D_7   D_8    D_9  D_10
S_1           0.0    0.0    0.0    0.0    0.0    0.0    0.0   0.0  110.0   0.0
S_2           0.0    0.0   20.0    0.0    0.0   60.0    0.0   0.0    0.0   0.0
S_3           0.0    0.0    0.0    0.0    0.0    0.0  100.0   0.0    0.0   0.0
S_4           0.0  100.0    0.0    0.0    0.0  140.0    0.0   0.0    0.0   0.0
S_5           0.0    0.0    0.0   60.0    0.0   40.0    0.0   0.0    0.0   0.0
S_6           0.0    0.0    0.0    0.0  180.0    0.0   30.0   0.0    0.0  70.0
S_7           0.0    0.0  130.0    0.0    0.0    0.0    0.0   0.0    0.0   0.0
Var Penalty  90.0    0.0    0.0  130.0    0.0    0.0   80.0  90.0   50.0   0.0


The optimized minmum total cost is therefore $21,980


### <font color=#FF0000> **Insight:** </font>
*Is the optimal allocation the same as Problem 2? Why?*


The optimal allocation is not the same as the previous problems. The alloation has changed because this problem has many new variables compared to previous problems. In the previous problems the penalty cost was a fixed cost so the only new variable was the price of the penalty, however with variable costs for non-delivery the problem may have a different optimized solution where the cost of not delivering may be lower than the cost of delivering for a supply center.

**Problem 6:**

Again, suppose not supplying milk to a demand market is costless (like Problem 2,) but the firm is still committed to supplying all the available milk. This time, the firm decides to supply the milk so that each demand center receives at least 50% of its demand. Formulate the problem of minimizing the total transportation costs using Gurobi and solve it. Then, report the optimal allocation and the minimum cost. 

Compare the optimal cost with your answer to Problem 2. Is it more or less? Why?

Suppose the firm sets this bar to 90% - each demand center should receive at least 90% of its demand. Is it possible? What is the maximum possible percentage?

In [20]:
# Your solution for Problem 6:
Cost6 = [[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
        [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
        [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
        [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
        [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
        [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
        [12, 19,  5, 28, 47, 39, 15, 35, 9, 51],
        [0 ,  0,  0,  0,  0,  0,  0,  0, 0, 0]]


SupplyCenter6 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7',]
Supply6 = [110, 80, 100, 240, 100, 280, 130, 0]
DemandMarket6 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand6 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd
import copy

#suppress console output
env = gb.Env(empty=True)
env.setParam("OutputFlag",0)
env.start()


#set function for repeated attempts
def Re_optimize6(SupplyCenter,Supply,DemandMarket,Demand,probn,Cost,perc_demand):
        
    #create variables to define matrix of variables
    Slen=len(Supply)
    Dlen=len(Demand)
    
    
    #Create variables for each demand and supply
    X6=probn.addVars(Slen,Dlen,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter for j in DemandMarket])
    
    
    #set objective function
    #since the penalty for not delivering is 0 then there is no additional facor in the objective function that is required
    probn.setObjective(sum(Cost[i][j]*X6[i,j] for i in range(Slen)for j in range (Dlen)),GRB.MINIMIZE)

    #set constraints
    ##setting the supply constrains for each supply centre
    for i in range(Slen):
        probn.addConstr(sum(X6[i,j] for j in range (Dlen))==Supply[i])
    
    ##setting the demand constrains for each demand centre with no over supply of demand
    for j in range(Dlen):
        probn.addConstr(sum(X6[i,j] for i in range (Slen))>=(Demand[j]*perc_demand))
    
    #constrani the upper bound of the demand no over supply
    for j in range(Dlen):
        probn.addConstr(sum(X6[i,j] for i in range (Slen))<=(Demand[j]))
        
    #ensure all supply is delivered
    #for i in range(Slen):

    #optimize problem
    probn.optimize()
    
    #set up matrix to print out results
    pbndict6={}
    l6=[]
    for i in X6:
        col=("D_"+str(i[1]+1))
        if col in pbndict6.keys():
            pbndict6[col].append(X6[i].X)
        else:
            l6=[]
            pbndict6[col]=l6
            pbndict6[col].append(X6[i].X)   
    df=pd.DataFrame(pbndict6,index=SupplyCenter)
    
    return df,probn



#setup problem
prob_6a=gb.Model("Question 6",env=env)
df6a,prob_6a=Re_optimize6(SupplyCenter6,Supply6,DemandMarket6,Demand6,prob_6a,Cost6,0.5)


print("\n")
print("\n")
print("The optimized supply and demand is visualized in a table below:")
print(df6a)

#print objective cost
#since the penalty for not delivering is 0 then there is no additional facor in the objective function that is required
print("\n")
print("The optimized minmum total cost, for delivering at least 50% of the demand therefore ${:,}".format(round(prob_6a.objVal)))






The optimized supply and demand is visualized in a table below:
            D_1    D_2   D_3   D_4    D_5    D_6    D_7   D_8    D_9  D_10
S_1         0.0    0.0   0.0   0.0    0.0    0.0    0.0   0.0  110.0   0.0
S_2         0.0    0.0  65.0   0.0    0.0    0.0    0.0  15.0    0.0   0.0
S_3         0.0    0.0   0.0   0.0    0.0    0.0  100.0   0.0    0.0   0.0
S_4         0.0  100.0   0.0   0.0    0.0  135.0    5.0   0.0    0.0   0.0
S_5         5.0    0.0   0.0  95.0    0.0    0.0    0.0   0.0    0.0   0.0
S_6         0.0    0.0   0.0   0.0  180.0    0.0    0.0  30.0    0.0  70.0
S_7        40.0    0.0  85.0   0.0    0.0    0.0    0.0   0.0    5.0   0.0
Dummy_Var   0.0    0.0   0.0   0.0    0.0    0.0    0.0   0.0    0.0   0.0


The optimized minmum total cost, for delivering at least 50% of the demand therefore $11,605


### <font color=#FF0000> **Insight:** </font>
*Compare the optimal cost with your answer to Problem 2. Is it more or less? Why?*

The optimal cost has **decreased** compared to problem #2 since the firm is committed to supplying only 50% of the cost then the model can supply 50% for the morre expensive demand ceters and more than 50% for the les expensive demand ceters


*Suppose the firm sets this bar to 90% - each demand center should receive at least 90% of its demand. Is it possible? What is the maximum possible percentage?*


Since supplier \#8 is not operational 440 of the total 1,480  is not available to meet demand.
In other words only 70.27\% of the total supply is available to meet demand, which is below 90\%.
Therefore it is not possible to supply 90\% of the demand
The maximum percentage possible is: 70.27\%

**Problem 7:**

Suppose the cost of not supplying each tonne of milk to each demand center is 70\\$/tone. As a short-term solution, the firm can ask one of the seven remaining suppliers to add 100 tonnes of milk to its supplying capacity by paying the selected supplier 1000\\$. Which supplier should the firm ask for this capacity boost to minimize costs? What is the change in the costs due to this boost? Can you solve this problem without redefining the problem for each supplier?

In [21]:
# Your solution for Problem 7:
#use shardow price
Cost7 = [[20, 49, 16, 30,  8, 35, 21, 40, 10, 12],
        [15, 53, 7, 20, 47, 11, 16, 17, 15, 44],
        [22, 25, 42, 22, 31, 9, 11, 29, 20, 5],
        [45,  6, 33, 35, 49, 25, 30, 47, 32, 31],
        [9, 12, 41, 15, 38, 14, 53, 22, 12, 13],
        [21, 24, 32, 49, 5, 47, 30, 23, 37, 8],
        [12, 19,  5, 28, 47, 39, 15, 35, 9, 51]]

Cost7.append([70, 70, 70, 70, 70, 70, 70, 70, 70, 70])


SupplyCenter7 = ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8']
Supply7 = [110, 80, 100, 240, 100, 280, 130, 440]
DemandMarket7 = ['D_1', 'D_2', 'D_3', 'D_4', 'D_5', 'D_6', 'D_7', 'D_8', 'D_9', 'D_10']
Demand7 = [90, 100, 150, 190, 180, 240, 210, 90, 160, 70]

# Your solution for Problem 1:
#import packages
import gurobipy as gb
from gurobipy import *
import pandas as pd


#setup problem
prob_7=gb.Model("Question 7")
prob_7.Params.LogToConsole = 0

#create variables to define matrix of variables
Slen7=len(Supply7)
Dlen7=len(Demand7)

#Create variables for each demand and supply
X7=prob_7.addVars(Slen7,Dlen7,lb=0,vtype=GRB.CONTINUOUS, name = [i+"_to_"+j for i in SupplyCenter7 for j in DemandMarket7])
#set objective function 
prob_7.setObjective(sum(Cost7[i][j]*X7[i,j] for i in range(Slen7)for j in range (Dlen7)),GRB.MINIMIZE)

#set constraints
##setting the supply constrains for each supply centre
for i in range(Slen7):
    prob_7.addConstr(sum(X7[i,j] for j in range (Dlen7))<=Supply7[i])
    
##setting the demand constrains for each demand centre with no over supply of demand
for j in range(Dlen7):
    prob_7.addConstr(sum(X7[i,j] for i in range (Slen7))>=Demand7[j])

#optimize
prob_7.optimize()

#
prob_7.Params.LogToConsole = 1
prob_7.printAttr(['Sense', 'Slack', 'Pi', 'RHS', 'SARHSLow', 'SARHSUp'])

#The new cost is the objective plus a cost of $1000 paid to a supplier - $5,900 (the shardow price for additional 100 units bysupplier 3)
#the shadow price accounts for the reallocation of units from other suppliers
Decreased_cost=-1000+59*100
print("The overall cost decreased by ${:,}".format(round(Decreased_cost)))

Set parameter LogToConsole to value 1

  Constraint        Sense        Slack           Pi          RHS     SARHSLow      SARHSUp 
------------------------------------------------------------------------------------------
          R0            <            0          -58          110          110          150 
          R1            <            0          -57           80           80          160 
          R2            <            0          -59          100          100          290 
          R3            <            0          -43          240          240          320 
          R4            <            0          -56          100          100          140 
          R5            <            0          -47          280          280          340 
          R6            <            0          -59          130          130          190 
          R7            <            0            0          440          440          inf 
          R8            >            0    

### <font color=#FF0000> **Insight:** </font>
 
*Which supplier should the firm ask for this capacity boost to minimize costs?*

Based on the sensitivity analysis, the shadow price for the (Pi), the lowest shadwoprice is -59.0 for supplier #3 and #7.
However only supplier #3 has an allowable increase of more than 100 units, whereas supplier #7 does not.
Therefore the firm should ask supplier #3 to boost their capacity


*What is the change in the costs due to this boost?*

The overall cost for the additional 100 units is decreased by \\$4,900


*Can you solve this problem without redefining the problem for each supplier?*

The sensitivity analysis provides the tools to allow evaluation of these criteria without need to redefine the problem

